# 02 — Session Splits, Sequence Windows and Baselines

A world model learns from sequences, but the evaluation protocol decides whether the reported performance is honest.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT / 'src'))

In [ ]:
from apexsim.config import load_config
from apexsim.data.features import Standardizer
from apexsim.data.windows import split_sessions, TelemetryWindowDataset
from apexsim.contracts import MODEL_INPUT_COLUMNS, TARGET_COLUMNS

config = load_config(ROOT/'configs/fast.yaml')
frame = pd.read_csv(ROOT/'artifacts/runs/reference_gru/canonical_telemetry.csv')
splits = split_sessions(frame, config.data.train_fraction, config.data.val_fraction, config.seed)
splits

## Why split by session?

Random frame splitting leaks nearly identical neighbouring frames into train and test. The model then appears to generalize while merely recognizing the same weather, driver and track episode. A session holdout asks a harder and more operational question: can the model roll forward in an unseen session?

In [ ]:
train_frame = frame[frame.session_id.isin(splits['train'])]
scaler = Standardizer.fit(train_frame)
dataset = TelemetryWindowDataset(frame, splits['train'], 32, 8, scaler, max_windows=100)
sample = dataset[0]
{k: (tuple(v.shape) if hasattr(v, 'shape') else v) for k, v in sample.items()}

## Visual window

```text
observed history                         predicted future
|<------------- 32 frames ------------->|<--- 8 frames --->|
[state + action + context] ...           [target states]
```

The model may look at future **actions/context** during controlled simulation because a scenario specifies them. It must not look at future target states.

In [ ]:
speed = scaler.inverse_targets(sample['state_history'].numpy())[:,0] * 3.6
plt.figure(figsize=(9,3))
plt.plot(speed)
plt.title('Observed history speed')
plt.xlabel('History step'); plt.ylabel('km/h'); plt.show()

## Baselines are scientific controls

Before a deep model, test persistence and linear transitions. A deep model that cannot beat a simple baseline at the operational horizon has not earned production complexity.